## 1. Foundational & Oracle Algorithms
These algorithms evaluate black-box functions (oracles) to determine global properties using fewer queries than classical methods, establishing the theoretical foundations of quantum speedups.

### 1.[Deutsch–Jozsa Algorithm](https://en.wikipedia.org/wiki/Deutsch%E2%80%93Jozsa_algorithm)

* **Problem Domain:** [Boolean Function Analysis](https://en.wikipedia.org/wiki/Analysis_of_Boolean_functions).

* **Core Function:** Determines if a black-box Boolean function $f: \{0,1\}^n \to \{0,1\}$ is `constant` ($0$ on all inputs or $1$ on all inputs) or `balanced` ($1$ for exactly half of the input domain and $0$ for the other half) in a single query.

* **Algorithmic Mechanism:**
    1. Initialize $n$ query qubits in state $\vert{}0\rangle$ and an ancilla qubit in state $\vert{}1\rangle$.
    2. Apply Hadamard gates ($H^{\otimes (n+1)}$) to create an initial uniform superposition.
    3. Query the oracle $U_f$, which maps $\vert{}x\rangle\vert{}y\rangle \to \vert{}x\rangle\vert{}y \oplus f(x)\rangle$, causing phase kickback.
    4. Apply $H^{\otimes n}$ to the query qubits and measure: if all measured bits are $0$, the function is constant; otherwise, it is balanced.

**Example:**

It is a specific algorithm that checks if function is constant or balanced in single query.

In [9]:
from qiskit import QuantumCircuit

# 2-qubit query register + 1 ancilla (Balanced Oracle for f(x) = x_0 XOR x_1)
qc = QuantumCircuit(3, 2)

# Step 1: Initialize states (|00> and |1>)
qc.x(2)
qc.barrier()

# Step 2: Apply Hadamards
qc.h([0, 1, 2])
qc.barrier()

# Step 3: Balanced Oracle implementation
qc.cx(0, 2)
qc.cx(1, 2)
qc.barrier()

# Step 4: Apply Hadamards & Measure
qc.h([0, 1])
qc.barrier()
qc.measure([0, 1], [0, 1])

print("Deutsch-Jozsa Circuit:")
print(qc.draw('text'))

Deutsch-Jozsa Circuit:
           ░ ┌───┐ ░            ░ ┌───┐ ░ ┌─┐   
q_0: ──────░─┤ H ├─░───■────────░─┤ H ├─░─┤M├───
           ░ ├───┤ ░   │        ░ ├───┤ ░ └╥┘┌─┐
q_1: ──────░─┤ H ├─░───┼────■───░─┤ H ├─░──╫─┤M├
     ┌───┐ ░ ├───┤ ░ ┌─┴─┐┌─┴─┐ ░ └───┘ ░  ║ └╥┘
q_2: ┤ X ├─░─┤ H ├─░─┤ X ├┤ X ├─░───────░──╫──╫─
     └───┘ ░ └───┘ ░ └───┘└───┘ ░       ░  ║  ║ 
c: 2/══════════════════════════════════════╩══╩═
                                           0  1 


### 2. [Bernstein–Vazirani Algorithm](https://en.wikipedia.org/wiki/Bernstein%E2%80%93Vazirani_algorithm)

* **Problem Domain:** Hidden String Retrieval.
* **Core Function:** Finds an $n$-bit hidden string $s \in \{0,1\}^n$ from an oracle $f(x) = s \cdot x \pmod 2$ in $\mathcal{O}(1)$ queries.
* **Algorithmic Mechanism:**
    1. Prepare query qubits in state $\vert{}0\rangle^{\otimes n}$ and ancilla in state $\vert{}-\rangle$.
    2. Apply Hadamard gates across all query qubits.
    3. Query the oracle, which applies $CX$ gates between query bit $x_i$ and the ancilla whenever $s_i = 1$.
    4. Apply Hadamard gates to the query qubits and measure to read out $s$ directly.

**Example**

We can check the structure itself.

In [8]:
from qiskit import QuantumCircuit

# Hidden bitstring s = "11" (2 query qubits + 1 ancilla)
qc = QuantumCircuit(3, 2)

# State Prep
qc.x(2)
qc.barrier()
qc.h([0, 1, 2])
qc.barrier()

# Oracle for s = "11"
qc.cx(0, 2)
qc.cx(1, 2)
qc.barrier()

# Readout Phase
qc.h([0, 1])
qc.barrier()
qc.measure([0, 1], [0, 1])

print("Bernstein-Vazirani Circuit (s='11'):")
print(qc.draw('text'))

Bernstein-Vazirani Circuit (s='11'):
           ░ ┌───┐ ░            ░ ┌───┐ ░ ┌─┐   
q_0: ──────░─┤ H ├─░───■────────░─┤ H ├─░─┤M├───
           ░ ├───┤ ░   │        ░ ├───┤ ░ └╥┘┌─┐
q_1: ──────░─┤ H ├─░───┼────■───░─┤ H ├─░──╫─┤M├
     ┌───┐ ░ ├───┤ ░ ┌─┴─┐┌─┴─┐ ░ └───┘ ░  ║ └╥┘
q_2: ┤ X ├─░─┤ H ├─░─┤ X ├┤ X ├─░───────░──╫──╫─
     └───┘ ░ └───┘ ░ └───┘└───┘ ░       ░  ║  ║ 
c: 2/══════════════════════════════════════╩══╩═
                                           0  1 


### 3. [Simon’s Algorithm](https://en.wikipedia.org/wiki/Simon%27s_problem)
* **Problem Domain:** Periodicity & Hidden XOR Subgroups.
* **Core Function:** Finds a hidden period bitstring $s \in \{0,1\}^n$ for a function $f(x) = f(y) \iff x \oplus y \in \{0, s\}$ in $\mathcal{O}(n)$ queries.
* **Algorithmic Mechanism:**
    1. Prepare two $n$-qubit registers ($\vert{}0\rangle^{\otimes n}\vert{}0\rangle^{\otimes n}$).
    2. Apply Hadamard transform to the first register.
    3. Query oracle $U_f$ to evaluate $f(x)$ into the second register.
    4. Apply Hadamard transform to the first register and measure it to obtain linear equations $y \cdot s = 0 \pmod 2$.
    5. Solve the resulting system of $n-1$ linear equations classically.

**Example:**

Similar to Deutsch-Jozsa and Simon's algorithms we can check the structure of algorithm itself.

In [12]:
from qiskit import QuantumCircuit

# Simon's algorithm circuit template (2-qubit registers for s = "11")
qc = QuantumCircuit(4, 2)

# Step 1: Superposition on Input Register
qc.h([0, 1])

# Step 2: Oracle for s = "11"
qc.cx(0, 2)
qc.cx(1, 2)
qc.cx(0, 3)
qc.cx(1, 3)
qc.barrier()

# Step 3: Hadamard on Input Register & Measure
qc.h([0, 1])
qc.barrier()
qc.measure([0, 1], [0, 1])

print("Simon's Algorithm Circuit Structure:")
print(qc.draw('text'))

Simon's Algorithm Circuit Structure:
     ┌───┐                     ░ ┌───┐ ░ ┌─┐   
q_0: ┤ H ├──■─────────■────────░─┤ H ├─░─┤M├───
     ├───┤  │         │        ░ ├───┤ ░ └╥┘┌─┐
q_1: ┤ H ├──┼────■────┼────■───░─┤ H ├─░──╫─┤M├
     └───┘┌─┴─┐┌─┴─┐  │    │   ░ └───┘ ░  ║ └╥┘
q_2: ─────┤ X ├┤ X ├──┼────┼───░───────░──╫──╫─
          └───┘└───┘┌─┴─┐┌─┴─┐ ░       ░  ║  ║ 
q_3: ───────────────┤ X ├┤ X ├─░───────░──╫──╫─
                    └───┘└───┘ ░       ░  ║  ║ 
c: 2/═════════════════════════════════════╩══╩═
                                          0  1 


### 4. [Hidden Subgroup Problem (HSP)](https://en.wikipedia.org/wiki/Hidden_subgroup_problem)
* **Problem Domain:** [Group Theory](https://en.wikipedia.org/wiki/Group_theory), [Abstract Algebra](https://en.wikipedia.org/wiki/Abstract_algebra), [Cryptography](https://en.wikipedia.org/wiki/Cryptography).
* **Core Function:** Identifies a hidden subgroup $H \le G$ of a finite group $G$ using oracle access to a function $f: G \to X$ that is constant and distinct on cosets of $H$.
* **Algorithmic Mechanism:**
    1. Prepare a quantum state in equal superposition over group elements $\vert{}G\rangle = \frac{1}{\sqrt{\vert{}G\vert{}}}\sum_{g \in G}\vert{}g\rangle\vert{}0\rangle$.
    2. Query the oracle $U_f$ to evaluate $\vert{}g\rangle\vert{}f(g)\rangle$.
    3. Measure the second register, collapsing the first register into a uniform superposition over a random coset $gH$.
    4. Apply the Fourier Transform over group $G$ to extract elements in the dual group or normalizer $H^\perp$, revealing $H$ through classical post-processing.
    
* **Connection:** Deutsch-Jozsa, Bernstein-Vazirani, Simon's, and Shor's algorithms are all special abelian instances of the general HSP framework.

In [2]:
from qiskit import QuantumCircuit

# Conceptual HSP implementation over Z_2 x Z_2 (simulating a hidden subgroup H = {00, 11})
qc = QuantumCircuit(4, 2)

# Step 1: Superposition over group G elements
qc.h([0, 1])

# Step 2: Oracle U_f mapping cosets of H to distinct values
qc.cx(0, 2)
qc.cx(1, 2)
qc.barrier()

# Step 3: Measurement of target register collapses state to coset gH (implicit)

# Step 4: Quantum Fourier Transform over Z_2 x Z_2 (Hadamard gates)
qc.h([0, 1])
qc.measure([0, 1], [0, 1])

print("Abelian HSP Circuit (Z_2 x Z_2 Group):")
print(qc.draw('text'))

Abelian HSP Circuit (Z_2 x Z_2 Group):
     ┌───┐           ░ ┌───┐┌─┐   
q_0: ┤ H ├──■────────░─┤ H ├┤M├───
     ├───┤  │        ░ ├───┤└╥┘┌─┐
q_1: ┤ H ├──┼────■───░─┤ H ├─╫─┤M├
     └───┘┌─┴─┐┌─┴─┐ ░ └───┘ ║ └╥┘
q_2: ─────┤ X ├┤ X ├─░───────╫──╫─
          └───┘└───┘ ░       ║  ║ 
q_3: ────────────────░───────╫──╫─
                     ░       ║  ║ 
c: 2/════════════════════════╩══╩═
                             0  1 
